In [8]:
# 초심자 핵심: RAG에 필요한 도구(SentenceTransformer: 문장 임베딩, faiss: 초고속 검색)를 가져오는 단계다.

import os
import pandas as pd
import numpy as np
import pickle
import google.generativeai as genai
from sentence_transformers import SentenceTransformer
import faiss
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY

# 전역 설정
EMBED_MODEL     = "paraphrase-multilingual-MiniLM-L12-v2" # 한국어 지원 잘되는 모델
TOP_K           = 5       # 유사 사례 5개 찾기
DATA_PATH       = "./data/GeoPulse_Final_Dataset_KOREAN.csv"
INDEX_PATH      = "rag/faiss_index.bin"
META_PATH       = "rag/metadata.pkl"

os.makedirs("rag", exist_ok=True)

In [9]:
# 초심자 핵심: 텍스트를 숫자로 바꿔서 '지도'처럼 저장하는 클래스다. 나중에 비슷한 분쟁을 이 지도의 거리를 보고 찾는다.
class ConflictVectorStore:
    def __init__(self):
        # 문장을 숫자로 바꿔주는 모델 로드 (최초 실행 시 다운로드됨)
        self.embedder = SentenceTransformer(EMBED_MODEL)
        self.index    = None
        self.metadata = []

    def _make_doc(self, row) -> str:
        # 분쟁 데이터를 하나의 검색용 문장으로 합치는 과정 (검색 품질 결정함)
        return (
            f"발생지: {row.get('발생지', '')} | 교전A: {row.get('정부군(교전A)', '')} | "
            f"교전B: {row.get('반군/적대측(교전B)', '')} | 원인: {row.get('분쟁원인', '')} | "
            f"강도: {row.get('전쟁강도', '')} | 연도: {row.get('연도', '')}"
        )

    def build(self, csv_path: str):
        # CSV 로드 -> 임베딩(숫자로 변환) -> FAISS 저장
        df = pd.read_csv(csv_path)
        docs = [self._make_doc(row) for _, row in df.iterrows()]
        
        print(f"🚀 {len(docs)}개 문서 임베딩 중... 잠시만 기다려라.")
        embeddings = self.embedder.encode(docs, convert_to_numpy=True, normalize_embeddings=True)

        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim) # 코사인 유사도 검색용 인덱스
        self.index.add(embeddings.astype(np.float32))

        self.metadata = df.to_dict(orient="records")
        faiss.write_index(self.index, INDEX_PATH)
        with open(META_PATH, "wb") as f:
            pickle.dump(self.metadata, f)
        print("✅ 벡터 저장소 구축 완료!")

    def load(self):
        self.index = faiss.read_index(INDEX_PATH)
        with open(META_PATH, "rb") as f:
            self.metadata = pickle.load(f)
        print("✅ 기존 인덱스 로드 완료!")

    def search(self, query: str, top_k: int = TOP_K):
        q_vec = self.embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
        scores, indices = self.index.search(q_vec, top_k)
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx < len(self.metadata):
                item = dict(self.metadata[idx])
                item["similarity"] = float(score)
                results.append(item)
        return results

In [15]:
# 초심자 핵심: Gemini(LLM)한테 검색된 데이터랑 DL 결과값을 주고 '전문가처럼 글 써라'라고 시키는 부분이다.
class GeoReportGenerator:
    def __init__(self):
        genai.configure(api_key=GEMINI_API_KEY)
        self.model = genai.GenerativeModel("gemini-2.5-flash")

    def generate(self, data_dict):
        # 프롬프트 엔지니어링 (니 리포트의 퀄리티를 결정함)
        prompt = f"""당신은 지정학 전문가입니다. 다음 데이터를 바탕으로 전문 리포트를 작성하세요.
        - 분쟁명: {data_dict['name']}
        - DL 예측 원인: {data_dict['dl_cause']}
        - DL 예측 위험도: {data_dict['dl_risk']}
        - 유사 역사적 사례: {data_dict['similar_cases']}
        
        섹션별로 분석을 수행하고 마지막엔 종합 전망을 작성하세요."""
        
        response = self.model.generate_content(prompt)
        return response.text

In [16]:
# 이제 위에서 만든 기능들을 하나로 합친다.
vector_store = ConflictVectorStore()

# 처음 한 번만 build(True)로 실행, 그 후엔 False로 로드만 하면 됨
vector_store.build(DATA_PATH) 

report_gen = GeoReportGenerator()

🚀 1586개 문서 임베딩 중... 잠시만 기다려라.
✅ 벡터 저장소 구축 완료!


In [17]:
# 하드코딩 말고 나중에 여기서 input()으로 받을 부분이다.
query = "러시아 우크라이나 돈바스 영토 분쟁"

# 1. 유사 사례 검색 (RAG)
similar = vector_store.search(query)

# 2. 리포트 생성 (DL 결과값은 이전 세션 결과라고 가정하고 임시로 넣음)
test_data = {
    "name": "러시아-우크라이나 분쟁",
    "dl_cause": "영토",
    "dl_risk": "전면전",
    "similar_cases": similar
}

report = report_gen.generate(test_data)
print("\n--- 📝 최종 분석 리포트 ---")
print(report)


--- 📝 최종 분석 리포트 ---
## 지정학 전문 리포트: 러시아-우크라이나 분쟁 심층 분석

**1. 서론**

러시아-우크라이나 분쟁은 21세기 국제 지정학적 환경에서 가장 중요하고 파괴적인 갈등 중 하나입니다. 본 보고서는 제공된 데이터, 특히 인공지능(DL)의 예측 모델과 유사 역사적 사례를 바탕으로 분쟁의 근본 원인, 현재 위험도, 그리고 향후 전개 양상을 지정학적 관점에서 심층 분석합니다.

**2. DL 예측 분석**

DL 모델은 러시아-우크라이나 분쟁에 대해 두 가지 핵심적인 예측을 제시했습니다.

*   **DL 예측 원인: 영토**
    DL 모델이 분쟁의 주요 원인을 '영토'로 지목한 것은 매우 통찰력 있는 분석입니다. 이는 러시아의 크림반도 강제 합병과 동부 돈바스 지역 점령 시도, 그리고 이를 넘어 우크라이나 전체 영토 주권을 침해하려는 야욕과 우크라이나의 확고한 영토 보전 의지 간의 근본적인 충돌을 정확히 반영합니다. 역사적으로 영토 분쟁은 국가의 생존 및 정체성과 직결되는 문제이므로, 외교적 타협이 극도로 어렵고 최극단적 군사적 대립으로 이어지는 경우가 많습니다. DL 모델은 이 분쟁의 핵심 동력이 단순한 정치적 갈등을 넘어선 영토 주권이라는 실존적 차원에 있음을 정확히 짚어냈습니다.

*   **DL 예측 위험도: 전면전**
    DL 모델의 '전면전' 위험도 예측은 2022년 2월 러시아의 전면 침공 이후 실제로 전개된 상황과 완벽하게 일치합니다. 대규모 병력 및 첨단 무기 투입, 광범위한 도시 파괴와 민간인 학살, 그리고 양측의 국가 역량을 총동원하는 총력전 양상은 전면전의 전형적인 특징을 보여줍니다. 이 예측은 영토라는 근본적 원인이 해결되지 않을 경우 군사적 대결이 어떤 양상으로 치달을 수 있는지를 명확히 보여주는 결과입니다.

**3. 유사 역사적 사례 분석**

제공된 유사 역사적 사례들은 러시아-우크라이나 분쟁의 복합적인 성격과 장기화 가능성을 이해하는 데 중요한 통찰을 제공합니다.

*   **2022-20